# Hidden Markov Model
*Characterized by three matrices and two concepts*
## **Concepts**:
1. Hidden states ($S$)
2. Observations ($O$)

## Matrices
1. Transition probability matrix ($A$)
2. Emission probability matrix ($B$)
3. Initial state distribution matrix ($π$)

### 1. Transition probability matrix ($A$)
- Defines the probability of transitioning from one hidden state to another.
- $A_{ij} = P(S_t = s_j \mid S_{t-1} = s_i)$
  - The probability of transitioning from state $s_i$ to state $s_j$ at time step $t$.
- Each row represents transitions starting from a specific hidden state, so each row must sum to 1:
  $$\sum_{j=1}^{N} A_{ij} = 1 \quad \forall i$$

### 2. Emission probability matrix ($B$)
- Defines the conditional probability of emitting (generating) a specific observation given a hidden state.
- $B_{ik} = P(O_t = o_k \mid S_t = s_i)$
  - The probability of observing $o_k$ when the current hidden state is $s_i$.
- Each row represents emission probabilities for a given state, so each row must sum to 1:
  $$\sum_{k=1}^{M} B_{ik} = 1 \quad \forall i$$ (where $M$ is the total number of unique observations)

### 3. Initial state distribution matrix ($π$)
- Defines the probability distribution of the starting hidden state at the first time step $t=1$.
- $\pi_i = P(S_1 = s_i)$ : The probability that the initial state is $s_i$.
- Since the model must start in one of the states, the sum of all initial probabilities must be equal to 1:
  $$\sum_{i=1}^{N} \pi_i = 1$$






## HMM solve 3 fundamental problems
1. What is the probability that the model generates a *specific sequence of observations* ?
  - Solved through `forward algorithm`
2. Given a sequence of observations, what is the most likely sequence of hidden states that generated them?
  - `Viterbi algorithm`
3. How to learn `transition` and `emission` probabilities from real observation data?
  - `Baum-Welch Algorithm(Expectation Maximization/EM)`

*We are interested in 2 and 3*

# Viterbi Algorithm

The Viterbi algorithm finds the most likely sequence of hidden states for a sequence of observations in a Hidden Markov Model (HMM). This notebook implements the algorithm in log space to avoid numerical underflow, then runs a small weather example.

An HMM is specified by initial state probabilities, transition probabilities, and emission probabilities. Each probability array below follows the order in `states` and `observations`.

In [ ]:
import numpy as np

def viterbi(observations, states, start_probability, transition_probability, emission_probability):
    """Return the most likely hidden-state sequence and its log probability."""
    # Preflight tests
    observations = list(observations)
    states = list(states)
    if not observations:
        raise ValueError("observations must contain at least one item")
    if not states:
        raise ValueError("states must contain at least one state")

    n_states = len(states)
    n_steps = len(observations)
    start = np.asarray(start_probability, dtype=float)
    transition = np.asarray(transition_probability, dtype=float)
    emission = np.asarray(emission_probability, dtype=float)
    if start.shape != (n_states,):
        raise ValueError("start_probability must have one value per state")
    if transition.shape != (n_states, n_states):
        raise ValueError("transition_probability must be a square state-by-state matrix")
    if emission.shape[0] != n_states:
        raise ValueError("emission_probability must have one row per state")
    if np.any(start < 0) or np.any(transition < 0) or np.any(emission < 0):
        raise ValueError("probabilities cannot be negative")

    # Map each observation to its column in the emission matrix.
    observation_index = {observation: i for i, observation in enumerate(observation_symbols)}
    try:
        obs_indices = [observation_index[item] for item in observations]
    except (KeyError, TypeError) as exc:
        raise ValueError(f"unknown observation: {exc.args[0]}") from exc
    if emission.shape[1] != len(observation_symbols):
        raise ValueError("emission matrix columns must match observation_symbols")

    # log(0) is -inf, correctly making impossible paths have zero probability.
    with np.errstate(divide="ignore"):
        log_start = np.log(start)
        log_transition = np.log(transition)
        log_emission = np.log(emission)

    scores = np.full((n_steps, n_states), -np.inf)
    backpointers = np.zeros((n_steps, n_states), dtype=int)
    scores[0] = log_start + log_emission[:, obs_indices[0]]

    for t in range(1, n_steps):
        for current in range(n_states):
            candidates = scores[t - 1] + log_transition[:, current]
            backpointers[t, current] = np.argmax(candidates)
            scores[t, current] = (
                candidates[backpointers[t, current]]
                + log_emission[current, obs_indices[t]]
            )

    last_state = int(np.argmax(scores[-1]))
    best_log_probability = float(scores[-1, last_state])
    path_indices = [last_state]
    for t in range(n_steps - 1, 0, -1):
        path_indices.append(int(backpointers[t, path_indices[-1]]))
    path_indices.reverse()
    return [states[i] for i in path_indices], best_log_probability

## Code explanation
```py
  scores = np.full((n_steps, n_states), -np.inf)
  backpointers = np.zeros((n_steps, n_states), dtype=int)
  scores[0] = log_start + log_emission[:, obs_indices[0]]

  for t in range(1, n_steps):
      for current in range(n_states):
          candidates = scores[t - 1] + log_transition[:, current]
          backpointers[t, current] = np.argmax(candidates)
          scores[t, current] = (
              candidates[backpointers[t, current]]
              + log_emission[current, obs_indices[t]]
          )

  last_state = int(np.argmax(scores[-1]))
  best_log_probability = float(scores[-1, last_state])
  path_indices = [last_state]
  for t in range(n_steps - 1, 0, -1):
      path_indices.append(int(backpointers[t, path_indices[-1]]))
  path_indices.reverse()
  return [states[i] for i in path_indices], best_log_probability
```

Above section of code is the dynamic programming implementation of Viterbi Algorithm

- `scores` : number of observations(no. of steps($M$)) × number of states($N$)  
$$
\begin{array}{c|ccc}
 & S_1 & S_2 & .. & S_N \\
\hline
O_1 & 0.01 & 0.89 & .. & 0.44 \\
O_2 & 0.22 & 0.10 & .. & 0.55 \\
 : & : & : & : & : \\
O_M & 0.01 & 0.33 & .. & 0.66
\end{array}
$$

score matrix : states along rows, observations along columns
- Captures $P(St | O)$

$$
\begin{array}{cc}
  & \begin{array}{ccc} O_1 & O_2 & O_3 & O4 \end{array} \\
  \begin{array}{c} St_1 \\ St_2 \\ St_3 \end{array} &
  \begin{bmatrix}
    0.78 & 0.88 & 0.11 & 0.23\\
    0.11 & 0.10 & 0.14 & 0.12\\
    0.11 & 0.02 & 0.75 & 0.65
  \end{bmatrix}
\end{array}
$$

- `transition` : number of states × number of states
$$
\begin{array}{c|ccc}
 & S_1 & S_2 & .. & S_N \\
\hline
S_1 & 0.01 & 0.89 & .. & 0.44 \\
S_2 & 0.22 & 0.10 & .. & 0.55 \\
 : & : & : & : & : \\
S_N & 0.01 & 0.33 & .. & 0.66
\end{array}
$$
  - Maps **transition probabilities** between states
- `emission` : number of states × number of possible observations in observation superset
$$
\begin{array}{c|ccc}
 & O_1 & O_2 & .. & O_V \\
\hline
S_1 & 0.01 & 0.89 & .. & 0.44 \\
S_2 & 0.22 & 0.10 & .. & 0.55 \\
 : & : & : & : & : \\
S_N & 0.01 & 0.33 & .. & 0.66
\end{array}
$$
  - Maps **hidden states** to **observable symbols**
  - $P(O_k | S_i)$ : Probability of observation $O_k$ given state $S_i$


*NOTE*: Difference between $N$ and $M$
- $N$: Number of possible observations; Number of elements in the Set of all possible observations
  - Can be considered as the *Vocabulary size*. Set of possiblee observations are the vocabulary
  - Sequence of $M$ is sampled from set of $N$
- $M$: Number of observations in the sample
  - May not contain all possible observations
  - Many observations may repeat



### Steps
1. Initialize `scores[0]` ($P(S_i,O_0)$)
    ```python
    scores[0] = log_start + log_emission[:, obs_indices[0]]
    ```
    - `log_start` : Initial probability of states($π$)
    - `log_emission[:, obs_indices[0]]` : emission probability for $0^{th}$ observation
      - Very first column from the `emission` matrix
    - After the operation `scores[0]` contain $P(S_i,O_0)$
      - According to definition of `emission` matrix:
        - $P(O_k|S_i) = (i,k)^{th}$ element of `emission` matrix
      - $P(S_i) × P(O_0 | S_i) = P(S_i, O_0)$
    
    - Observe closely; We've the luxuary to do entire column operation, because we already knew `log_start`.
2. This is a combination of 2 steps:
    1. Calculate the best candidates for next `log_start` equivalent; Lets call it $S_{current}$.   
    2. Multiply them with emission probability
    - Instead of implementing them as two decoupled row operations, we combine them inside the same loop
    ```python
      candidates = scores[t - 1] + log_transition[:, current]
      backpointers[t, current] = np.argmax(candidates)
      scores[t, current] = (
          candidates[backpointers[t, current]]
          + log_emission[current, obs_indices[t]]
      )
    ```
    - First statement calculate the initial probability for the $t^{th}$ step for the state `current`. Second statement captures the highest probability candidate from the set of candidates. This candidate is chosen as the predecessor of `current` state.
    - $3^{rd}$ statement caculate the `score` by multiplying with `emission` probability



In [ ]:
# Scratchpad
data = np.random.random((2,3))
print(data)
print("first row:")
print(data[0])
print(data.shape)

[[0.28317427 0.59186993 0.29601556]
 [0.24783677 0.71372695 0.01028304]]
first row:
[0.28317427 0.59186993 0.29601556]
(2, 3)


## Example: inferring weather from activities

The hidden states are `Rainy` and `Sunny`; the observations are `walk`, `shop`, and `clean`. The returned log probability is the log of the joint probability of the best hidden path and the observations.

In [ ]:
states = ["Rainy", "Sunny"]
observation_symbols = ["walk", "shop", "clean"]
observations = ["walk", "shop", "clean"]

start_probability = [0.6, 0.4]
transition_probability = [
    [0.7, 0.3],  # From Rainy to [Rainy, Sunny]
    [0.4, 0.6],  # From Sunny to [Rainy, Sunny]
]
emission_probability = [
    [0.1, 0.4, 0.5],  # Rainy: P(walk), P(shop), P(clean)
    [0.6, 0.3, 0.1],  # Sunny: P(walk), P(shop), P(clean)
]

best_path, log_probability = viterbi(
    observations,
    states,
    start_probability,
    transition_probability,
    emission_probability,
)
print("Most likely weather sequence:", best_path)
print("Log probability of this path and observations:", log_probability)

## How it works

1. Initialize each state's score using its start and first-observation probabilities.
2. At each subsequent observation, choose for each current state the previous state that maximizes the accumulated log probability. Save that choice in a backpointer.
3. Select the highest-scoring final state and follow the backpointers backward to recover the best sequence.

The dynamic-programming computation takes O(T × N²) time and O(T × N) memory, where T is the number of observations and N is the number of states.